# 10 — MGSP (bài TAIMMA 2027) trên Google Colab A100

Chạy `experiments/stage8_mgsp.yaml`: **MGSP — Metadata-Guided Spatial Pooling**. Metadata sinh query, query chạy
cross-attention lên lưới patch token của **từng ảnh riêng** (clin, derm), dùng chung trọng số và chung query.
Mô hình đơn giản, tách khỏi bài Q1 (không MoE / gate / head long-tail / modality dropout / 384px).

| Run | Mô hình | Fold | Bảng |
|---|---|---|---|
| R1 | Chỉ ảnh: ConcatMLP `[CLS_c ‖ CLS_d] → MLP` | 5 | 2 |
| R2 | Nối metadata: ConcatMLP `[CLS_c ‖ CLS_d ‖ meta] → MLP` | 5 | 2 |
| **R4** | **MGSP đầy đủ** (2 query metadata + 2 query tĩnh, giữ checkpoint) | 5 | 2 |
| R3 | MGSP chỉ query tĩnh (không metadata dẫn hướng) | 0 | 3 |
| R5 | MGSP attention chỉ trên derm | 0 | 3 |
| R6 | MGSP bỏ CLS khỏi z | 0 | 3 |
| R7 | MGSP gộp 2 ảnh (392 token, 1 softmax) | 0 | 3 |
| R8 | MGSP 4 + 4 query | 0 | 3 |

Chung: ViT-B/16 224px · CE + sampler q0.5 · squash + aug mạnh · lr 5e-5 (backbone ×1.0) · drop path 0.1 · dropout 0.4 ·
20 epoch · patience 5 · batch hiệu dụng 64. Thứ tự: **8 run fold 0 trước** (đủ Bảng 3 và so sánh R4 vs R2 sớm), rồi
fold 1–4 của R1, R2, R4. Thêm mô hình ML trên metadata 5 fold (CPU, ~2 phút) cho dòng "chỉ metadata" và late fusion.

**Thời gian ước tính (A100):** ~20 phút/run khi chạy một mình → 80GB (3 run song song): **~3.5–4 giờ** cho 20 run,
trong đó 8 run fold 0 xong sau ~1.5 giờ · 40GB (2 run song song): ~4 giờ · 2 máy (`HUONG="top"` và `"down"`): ~2 giờ.
Số đo thật in ra sau run đầu tiên.

**Cách chạy**
1. Runtime → Change runtime type → **A100 GPU**, bật **High-RAM**. Runtime → **Run all**, cho phép truy cập Drive.
2. Colab ngắt: mở lại và **Run all**. Run đã xong (đã đồng bộ lên Drive) được bỏ qua.
3. Máy thứ 2 (tuỳ chọn): mở thêm một bản notebook này, đặt `HUONG = "down"` → hai máy dùng chung `_claims/`, không chạy trùng.
4. Kết quả: `MyDrive/milk10k_mgsp/outputs/mgsp_tables/` (`mgsp_tables.md`, `mgsp_results.xlsx`, `submissions/`);
   checkpoint R4: `MyDrive/milk10k_mgsp/checkpoints/` (vẽ heatmap / counterfactual).

In [ ]:
# ===================== CẤU HÌNH =====================
REPO_URL = "https://github.com/trong5nhan6/fussion.git"
BRANCH = "main"
DRIVE_URL = "https://drive.google.com/file/d/1lM1nB_4pq3Vg-GwlaTMQjtus_bpbMhGX/view?usp=drive_link"  # zip dữ liệu MILK10k
DRIVE_DIR = "/content/drive/MyDrive/milk10k_mgsp"          # thư mục riêng của bài TAIMMA

# ===================== CHỌN THÍ NGHIỆM =====================
EXPERIMENT = "experiments/stage8_mgsp.yaml"
VARIANTS = ["R1", "R2", "R4", "R3", "R5", "R6", "R7", "R8"]   # bỏ bớt nếu thiếu thời gian (vd bỏ R8)
FOLDS = None             # None = theo yaml (R1/R2/R4: 5 fold, còn lại fold 0) | vd [0]: chỉ fold 0 cho mọi variant

HUONG = "top"            # top: chạy danh sách từ trên xuống | down: từ dưới lên (máy thứ 2)
TIME_BUDGET_H = 23.0     # không bắt đầu run mới nếu không kịp xong trước mốc này
JOBS_PER_GPU = "auto"    # auto: 3 nếu VRAM >= 70GB, 2 nếu 40GB (ViT-B 224px nhẹ) | hoặc số cụ thể (2 nếu bị CUDA out of memory)
TABULAR = ["logreg", "hgb", "catboost"]   # mô hình ML trên metadata, 5 fold (CPU)

# Dice test leaderboard của các file đã nộp (thư mục trong mgsp_tables/submissions) -> điền vào Bảng 2
TEST_SCORES = {
    # "vit_base__abl_R1__cv5": None,
    # "vit_base__abl_R4__cv5": None,
}

In [ ]:
# ===================== THAM SỐ TRAIN =====================
# Batch 16 x 4 lấy từ configs/env/colab_a100.yaml (không grad checkpointing).
PARAMS = {
    "train.amp_dtype": "bf16",
    "train.epochs": 20,
    "train.patience": 5,
    "seed": 42,
    "predict.tta": True,
}
RUN_SUFFIX = ""

## 1. Google Drive, GPU và cấu hình song song

In [ ]:
import os, subprocess, torch
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DRIVE_DIR + "/outputs", exist_ok=True)

!nvidia-smi --query-gpu=name,memory.total --format=csv
assert torch.cuda.is_available(), "Chưa bật GPU: Runtime -> Change runtime type -> A100 GPU"
props = torch.cuda.get_device_properties(0)
vram = props.total_memory / 2**30
if "A100" not in props.name:
    print(f"CẢNH BÁO: đang dùng {props.name}, không phải A100. bf16 cần GPU Ampere trở lên (A100, L4, H100).")
jobs = JOBS_PER_GPU if JOBS_PER_GPU != "auto" else (3 if vram >= 70 else 2 if vram >= 38 else 1)
cpus = os.cpu_count()
PARAMS["data.num_workers"] = max(2, cpus // jobs - 1)
GPUS = ",".join(["0"] * jobs)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # giảm phân mảnh VRAM
print(f"{props.name} {vram:.0f}GB | {cpus} CPU | {jobs} run song song | {PARAMS['data.num_workers']} worker/run")

## 2. Code và dữ liệu

In [ ]:
REPO_DIR = "/content/" + REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
DATA_DIR = "/content/MILK10k"            # ổ SSD của Colab: đọc ảnh nhanh hơn Drive
os.environ["MILK10K_ROOT"] = DATA_DIR
if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone --depth 1 -b {BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!pip install -q "timm>=1.0" gdown openpyxl catboost
!git log -1 --format="Commit: %h %s"
!python scripts/setup_data.py --drive-url "{DRIVE_URL}" --dest {DATA_DIR}

In [ ]:
# Kiểm tra nhanh toàn bộ pipeline, gồm MGSP trên ViT / Swin / ConvNeXt (~2–3 phút)
!python -m pytest tests -q -p no:cacheprovider

## 3. Mô hình ML trên metadata (5 fold, CPU) — dòng "chỉ metadata" và late fusion của Bảng 2

In [ ]:
import glob
if all(os.path.exists(f"{DRIVE_DIR}/outputs/ml_{m}__5fold/oof.csv") for m in TABULAR):
    print("Đã có trên Drive, bỏ qua.")
else:
    tab_models = " ".join(TABULAR)
    !python scripts/train_tabular.py --model {tab_models} --set "train.folds=[0,1,2,3,4]"
    !mkdir -p "{DRIVE_DIR}/outputs" && rsync -a outputs/ml_*__5fold "{DRIVE_DIR}/outputs/"

## 4. Đồng bộ kết quả lên Drive và danh sách run

In [ ]:
# Chạy tiếp: run đã xong trên Drive được chép về outputs/ và bỏ qua. --shared-dir: 2 máy không chạy trùng run.
restore_arg = (f"--restore {DRIVE_DIR}/outputs --shared-dir {DRIVE_DIR}/outputs"
               + (" --reverse" if HUONG == "down" else ""))
GPU_LOG = f"outputs/_gpu_usage_{HUONG}.csv"
os.makedirs("outputs", exist_ok=True)
# Đồng bộ nền mỗi 5 phút (bỏ checkpoint .pt — checkpoint R4 được chép riêng ở bước 6) + ghi mức dùng GPU mỗi phút
sync_cmd = f"while true; do rsync -a --exclude '*.pt' outputs/ '{DRIVE_DIR}/outputs/'; sleep 300; done"
if "sync_proc" not in globals() or sync_proc.poll() is not None:
    sync_proc = subprocess.Popen(["bash", "-c", sync_cmd])
    gpu_log = subprocess.Popen("nvidia-smi --query-gpu=timestamp,utilization.gpu,memory.used,power.draw "
                               f"--format=csv -l 60 > {GPU_LOG}", shell=True)
print("Đồng bộ nền:", DRIVE_DIR + "/outputs")

In [ ]:
import shlex, yaml

def to_set_args(params: dict) -> str:
    # Chuyển PARAMS thành chuỗi --set key=value cho scripts (bỏ qua giá trị None)
    items = []
    for k, v in params.items():
        if v is None:
            continue
        val = yaml.safe_dump(v, default_flow_style=True).strip().removesuffix("...").strip()
        items.append(shlex.quote(f"{k}={val}"))
    return ("--set " + " ".join(items)) if items else ""

set_arg = to_set_args(PARAMS)
suffix_arg = f"--suffix {RUN_SUFFIX}" if RUN_SUFFIX else ""
print("Override:", set_arg or "(không có)")
print("Hậu tố tên run:", RUN_SUFFIX or "(không có)")

In [ ]:
exp = yaml.safe_load(open(EXPERIMENT, encoding="utf-8"))
select_arg = f"--variants {','.join(VARIANTS)}" + (f" --folds {','.join(map(str, FOLDS))}" if FOLDS else "")
n_runs = sum(len(FOLDS or exp["variants"][v].get("folds", [0])) for v in VARIANTS)
MIN_PER_RUN = 20         # phút / run ViT-B 224px trên A100 khi chạy 1 mình (ước lượng)
speed = {1: 1.0, 2: 1.6, 3: 1.8}.get(jobs, 1.8)
print(f"{n_runs} run (trừ run đã có), ước tính ~{n_runs * MIN_PER_RUN / 60 / speed:.1f} giờ trên A100 "
      f"({jobs} run song song). Số đo thật in ra sau run đầu tiên.")
!python scripts/run_grid.py {EXPERIMENT} --env configs/env/colab_a100.yaml {select_arg} {set_arg} {suffix_arg} {restore_arg} --dry-run

## 5. Train

In [ ]:
!python scripts/run_grid.py {EXPERIMENT} --env configs/env/colab_a100.yaml {select_arg} {set_arg} {suffix_arg} {restore_arg} --gpus {GPUS} --est-run-h 0.35 --time-budget-h {TIME_BUDGET_H}

## 6. Lưu checkpoint R4 lên Drive (heatmap attention và counterfactual metadata)

In [ ]:
ck_dir = f"{DRIVE_DIR}/checkpoints"
for pt in sorted(glob.glob("outputs/vit_base__abl_R4__f*/fold*/best.pt")):
    run = pt.split("/")[1]
    dst = f"{ck_dir}/{run}"
    if not os.path.exists(f"{dst}/best.pt"):
        os.makedirs(dst, exist_ok=True)
        !cp "{pt}" "{dst}/best.pt" && cp "outputs/{run}/config.yaml" "{dst}/"
    print("->", dst)

## 7. Bảng kết quả và file Excel
Chạy 2 máy: đợi cả hai xong rồi chạy ô này ở một máy. Nộp `submissions/vit_base__abl_R1__cv5` và `..._R4__cv5` lên
leaderboard, điền điểm vào `TEST_SCORES` rồi **chỉ chạy lại ô này**.

In [ ]:
from IPython.display import Markdown, display
!rsync -a --exclude '*.pt' outputs/ "{DRIVE_DIR}/outputs/"
yaml.safe_dump({k: v for k, v in TEST_SCORES.items() if v is not None}, open("outputs/test_scores.yaml", "w"))
!python scripts/mgsp_tables.py --dirs "{DRIVE_DIR}/outputs" --out outputs/mgsp_tables --test-scores outputs/test_scores.yaml > /dev/null
display(Markdown(open("outputs/mgsp_tables/mgsp_tables.md", encoding="utf-8").read()))
!rsync -a outputs/mgsp_tables/ "{DRIVE_DIR}/outputs/mgsp_tables/"
print("Excel:", DRIVE_DIR + "/outputs/mgsp_tables/mgsp_results.xlsx")

In [ ]:
# Mức dùng GPU trong lúc train (lấy mẫu mỗi phút)
import pandas as pd
u = pd.read_csv(GPU_LOG, skipinitialspace=True)
util = u.filter(like="utilization").iloc[:, 0].str.rstrip(" %").astype(float)
mem = u.filter(like="memory.used").iloc[:, 0].str.rstrip(" MiB").astype(float) / 1024
print(f"GPU util trung bình {util.mean():.0f}% (trung vị {util.median():.0f}%), VRAM tối đa {mem.max():.1f}GB, {len(u)} mẫu")